In [7]:
import duckdb

In [22]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [34]:
df = con.execute("""
                SELECT *
                 FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) as Linha
                    FROM bronze_z0019
                    WHERE data_ingestao >= '2025-04-26'
                 ) WHERE Linha = 1
                 
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,Linha
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-04-26 09:13:46.217099,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-04-26 09:13:46.217099,1
2,10003,CHAVE INGLESA,BT10,100,50,z0019_2.csv,2026-04-26 09:34:20.905235,1
3,10004,ALICATE,BT20,100,200,z0019_2.csv,2026-04-26 09:34:20.905235,1
4,10005,SERROTE,BT30,100,0,z0019_2.csv,2026-04-26 09:34:20.905235,1


In [40]:
df.columns

Index(['NATBR', 'MAKTX', 'WERKS', 'MAINS', 'LABST'], dtype='object')

In [42]:
df_final = df.rename(columns={
    'NATBR': 'ID',
    'MAKTX': 'NM_PRODUTO',
    'WERKS': 'ID_CATEGORIA',
    'MAINS': 'ID_FORNECEDOR',
    'LABST': 'VL_PRECO'
})
df_final.head(10)

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10003,CHAVE INGLESA,BT10,100,50
3,10004,ALICATE,BT20,100,200
4,10005,SERROTE,BT30,100,0


In [44]:
df_final.dtypes

ID               object
NM_PRODUTO       object
ID_CATEGORIA     object
ID_FORNECEDOR    object
VL_PRECO         object
dtype: object

In [ ]:
df2 = df_final
df2 = df2.astype(
    {
        'ID': int,
        'NM_PRODUTO': str,
        'ID_CATEGORIA': str,
        'ID_FORNECEDOR': int,
        'VL_PRECO': float
    }


)


df2.dtypes


ID                 int64
NM_PRODUTO        object
ID_CATEGORIA      object
ID_FORNECEDOR      int64
VL_PRECO         float64
dtype: object

In [49]:
con.execute('''
CREATE TABLE IF NOT EXISTS produtos (
    id BIGINT,
    nm_produto TEXT,
    id_categoria TEXT,
    id_fornecedor BIGINT,
    vl_preco FLOAT
)
''')

In [50]:
df2.head(10)

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10003,CHAVE INGLESA,BT10,100,50.0
3,10004,ALICATE,BT20,100,200.0
4,10005,SERROTE,BT30,100,0.0


In [52]:
con.execute("INSERT INTO produtos SELECT * from df2")

In [53]:
df_resultado = con.execute("select * from produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10003,CHAVE INGLESA,BT10,100,50.0
3,10004,ALICATE,BT20,100,200.0
4,10005,SERROTE,BT30,100,0.0


In [54]:
con.close()